In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
folder = '/content/drive/MyDrive/TransformesCod/03-restaurant/Chatbot_RAG/'
QA_file = folder + 'QA_file.csv'
QAfaiss_file = folder + 'QAfaiss_file.bin'

In [14]:
import faiss
index = faiss.read_index(QAfaiss_file)

import pandas as pd
df = pd.read_csv(QA_file)
df.head(20)

,question,answer
0,ما هي المكونات اللازمة لتحضير بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
1,ما هي المكونات المطلوبة لعمل بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
2,مما يتكون بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
3,إيش هي مقادير بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
4,أعطني قائمة المحتويات لطبق بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
5,ما هو نوع الحمية الغذائية ل بالو شاهي؟,نباتي
6,هل بالو شاهي نباتي أم غير نباتي؟,نباتي
7,ما هو النظام الغذائي ل بالو شاهي؟,نباتي
8,هل يناسب بالو شاهي الأشخاص النباتيين؟,نباتي
9,تصنيف بالو شاهي من ناحية الدايت؟,نباتي


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v1')


def get_embeddings(text):
  embeddings = model.encode(text , convert_to_tensor=True)
  return embeddings.numpy()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

In [28]:
def get_answer(query , df , index):
  query_embedding = get_embeddings([query])
  faiss.normalize_L2(query_embedding)
  _ , indices = index.search(query_embedding , 1)
  answer = df.iloc[indices[0][0]]['answer']
  return answer


In [30]:
query = ' ماهو مذاق بالو شاهي؟'

s = get_answer(query , df , index)
print(s)

حلو
